# Tuần 2 — Tiền xử lý và hai mô hình đầu tiên
Thời lượng dự kiến: **3,5 giờ** (2,5 giờ notebook, 1 giờ viết nháp Mục 3.1–3.5 của báo cáo)

**Mục tiêu.** Sau tuần này, sinh viên có thể:
1. sử dụng scikit-learn với ba thao tác `fit`, `transform`, `predict`;
2. tiền xử lý dữ liệu dạng bảng đúng quy trình: mã hóa one-hot, chuẩn hóa, chỉ học từ train;
3. huấn luyện và so sánh E1 (hồi quy logistic) và E2 (kNN) trên validation;
4. đo được tác động của rò rỉ dữ liệu và đọc hệ số của hồi quy logistic.

**Cách dùng notebook.** Mỗi mục được đánh dấu bằng một nhãn:
`[KIẾN THỨC]` nội dung cần đọc · `[MINH HỌA]` ô mã có sẵn, chạy và quan sát kết quả ·
`[YÊU CẦU]` phần sinh viên thực hiện: thay mỗi chỗ `None` hoặc `____` bằng mã phù hợp ·
`[TỰ KIỂM TRA]` ô kiểm tra tự động, in **ĐẠT** khi kết quả đúng ·
`[BÁO CÁO]` ghi câu trả lời ngắn vào ô văn bản để đưa vào báo cáo ·
`[CÂU HỎI SUY NGẪM]` câu hỏi để sinh viên tự trả lời trước; gợi ý ở cuối notebook.

> **Sổ tay thư viện** (`00_so_tay_thu_vien.pdf`, phát kèm đồ án) — các mục cần tra cứu trong tuần này: mục 4 (chia dữ liệu, tránh rò rỉ); mục 5 (biến dữ liệu thành số); mục 6 (huấn luyện, dự đoán, ngưỡng); mục 7 (đánh giá mô hình); mục 10 (đọc thông báo lỗi). Mỗi mục có đoạn mã mẫu chạy trên chính dữ liệu của đồ án, kết quả in ra và các lỗi thường gặp.

## Thiết lập
`[MINH HỌA]` Ô dưới đây nạp thư viện, tải dữ liệu và chia thành ba tập. Mã này **do giảng viên cung cấp, không thuộc phần đánh giá**; sinh viên chạy ô này mỗi khi mở notebook (khoảng 15 giây ở lần đầu).

In [ ]:
import io, json, zipfile, hashlib, datetime, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)

# Dữ liệu UCI Bank Marketing, tệp bank-full.csv; mã SHA-256 bảo đảm mọi sinh viên dùng cùng một tệp
NGUON = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
SHA256 = "d1513ec63b385506f7cfce9f2c5caa9fe99e7ba4e8c3fa264b3aaf0f849ed32d"

def doc_du_lieu():
    """Đọc du_lieu/bank-full.csv; nếu chưa có thì tải từ UCI (tệp nén lồng hai lớp) và lưu lại."""
    tep = Path("du_lieu/bank-full.csv")
    if not tep.exists():
        tep.parent.mkdir(exist_ok=True)
        ngoai = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(NGUON, timeout=30).read()))
        trong = zipfile.ZipFile(io.BytesIO(ngoai.read("bank.zip")))
        tep.write_bytes(trong.read("bank-full.csv"))
    assert hashlib.sha256(tep.read_bytes()).hexdigest() == SHA256, "Tệp dữ liệu không đúng phiên bản của đồ án."
    df = pd.read_csv(tep, sep=";")
    return df.assign(ma_dong=[f"dong:{i + 1:05d}" for i in range(len(df))])

du_lieu = doc_du_lieu()
train_val, test = train_test_split(du_lieu, test_size=0.2, stratify=du_lieu["y"], random_state=42)
train, val = train_test_split(train_val, test_size=0.25, stratify=train_val["y"], random_state=42)
train, val, test = (d.reset_index(drop=True) for d in (train, val, test))
print("Kích thước train / validation / test:", len(train), len(val), len(test), "(theo đề: 27126 / 9042 / 9043)")

# Thuộc tính đầu vào của mô hình (đã loại duration — lý do ở tuần 1, mục 1.6)
THUOC_TINH_SO = ["age", "balance", "day", "campaign", "pdays", "previous"]
THUOC_TINH_PL = ["job", "marital", "education", "default", "housing", "loan", "contact", "month", "poutcome"]

def chi_so(y_that, y_du_doan):
    """Các chỉ số chính, với yes là lớp cần phát hiện (tính bằng scikit-learn)."""
    p, r, f, _ = precision_recall_fscore_support(y_that, y_du_doan, labels=["yes"], zero_division=0)
    return {"accuracy": accuracy_score(y_that, y_du_doan), "precision_yes": p[0], "recall_yes": r[0], "f1_yes": f[0]}

Kích thước train / validation / test: 27126 9042 9043 (theo đề: 27126 / 9042 / 9043)


`[YÊU CẦU]` Sinh viên điền mã số sinh viên. Các thí nghiệm tuần này dùng `random_state` cố định trong mã để kết quả có thể tái lập; `S_SV` chưa được dùng trong tuần 2.

In [ ]:
MSSV = "3124411148"          # YÊU CẦU: điền mã số sinh viên
S_SV = int(MSSV[-3:]) if MSSV[-3:].isdigit() else None
assert S_SV is not None, "CHƯA ĐẠT: cần điền MSSV rồi chạy lại ô này."
print("ĐẠT: seed cá nhân S_SV =", S_SV)

ĐẠT: seed cá nhân S_SV = 148


## 2.1 `[KIẾN THỨC]` Vì sao cần tiền xử lý
Các mô hình của scikit-learn chỉ nhận **số**. Dữ liệu có hai loại thuộc tính:
- **phân loại** (`job`, `month`, …): mã hóa **one-hot**, mỗi giá trị thành một cột 0/1. Ví dụ `contact` có 3 giá trị nên thành 3 cột `contact_cellular`, `contact_telephone`, `contact_unknown`;
- **số** (`age`, `balance`, …): **chuẩn hóa** $z = (x-\mu)/\sigma$ với $\mu$, $\sigma$ là trung bình và độ lệch chuẩn **tính trên train**. Nếu không chuẩn hóa, `balance` (hàng nghìn euro) lấn át `age` (vài chục năm) khi tính khoảng cách trong kNN.

| Lệnh | Ý nghĩa |
|---|---|
| `fit(...)` | **học** từ dữ liệu: danh sách giá trị của one-hot, $\mu$ và $\sigma$ của chuẩn hóa, tham số mô hình |
| `transform(...)` | **biến đổi** dữ liệu bằng những gì đã học |
| `predict(...)` | **dự đoán** nhãn |

**Nguyên tắc: chỉ `fit` trên train.**

`[MINH HỌA]` Tiền xử lý 4 khách ví dụ (dùng cho **Bảng 7**). Với `age` = 25, 40, 60, 35: $\mu = 40$, $\sigma = \sqrt{(15^2+0^2+20^2+5^2)/4} \approx 12{,}75$, nên khách 25 tuổi có $z = (25-40)/12{,}75 \approx -1{,}18$.

In [ ]:
vi_du = pd.DataFrame({"age": [25, 40, 60, 35], "balance": [100, 5000, 20000, 0], "contact": ["cellular", "telephone", "cellular", "unknown"]})
bo_vi_du = make_column_transformer((OneHotEncoder(), ["contact"]), (StandardScaler(), ["age", "balance"]), verbose_feature_names_out=False)
pd.DataFrame(bo_vi_du.fit_transform(vi_du), columns=bo_vi_du.get_feature_names_out()).round(2)

,contact_cellular,contact_telephone,contact_unknown,age,balance
0,1.0,0.0,0.0,-1.18,-0.76
1,0.0,1.0,0.0,0.00,-0.16
2,1.0,0.0,0.0,1.57,1.68
3,0.0,0.0,1.0,-0.39,-0.77


## 2.2 `[YÊU CẦU]` Tiền xử lý dữ liệu thật
Yêu cầu sinh viên thực hiện:
1. tạo bộ tiền xử lý `tien_xu_ly` bằng `make_column_transformer`: `OneHotEncoder(handle_unknown="ignore")` cho `THUOC_TINH_PL`, `StandardScaler()` cho `THUOC_TINH_SO`;
2. `fit_transform` trên train, lưu vào `X_train`; chỉ `transform` trên validation, lưu vào `X_val`.

In [ ]:
tien_xu_ly = make_column_transformer(
    (OneHotEncoder(handle_unknown="ignore"), THUOC_TINH_PL),
    (StandardScaler(), THUOC_TINH_SO), verbose_feature_names_out=False)    # YÊU CẦU: bước 1 (thêm tham số verbose_feature_names_out=False để tên cột gọn)
X_train = tien_xu_ly.fit_transform(train)        # YÊU CẦU: bước 2, train
X_val = tien_xu_ly.transform(val)        # YÊU CẦU: bước 2, validation

In [ ]:
# [TỰ KIỂM TRA]
assert X_train is not None and X_val is not None, "CHƯA ĐẠT: cần tạo X_train và X_val."
assert X_train.shape[0] == len(train) and X_val.shape[0] == len(val), "CHƯA ĐẠT: mỗi khách phải tương ứng một dòng."
chuan_hoa = tien_xu_ly.named_transformers_["standardscaler"]
assert np.allclose(chuan_hoa.mean_, train[THUOC_TINH_SO].mean()), "CHƯA ĐẠT: trung bình dùng để chuẩn hóa phải tính trên TRAIN."
print("ĐẠT: số khách x số đặc trưng:", X_train.shape, X_val.shape)
pd.Series(chuan_hoa.mean_, index=THUOC_TINH_SO).round(1).to_frame("trung bình học từ train").T

ĐẠT: số khách x số đặc trưng: (27126, 50) (9042, 50)


,age,balance,day,campaign,pdays,previous
trung bình học từ train,40.9,1368.1,15.8,2.8,40.1,0.6


`[CÂU HỎI SUY NGẪM]` (Câu 1) Vì sao số đặc trưng (50) lớn hơn số thuộc tính đầu vào (15)? (Câu 2) Nếu viết `X_val = tien_xu_ly.fit_transform(val)` thì sai ở đâu?

`Câu 1`: Vì 9 thuộc tính phân loại đã được mã hóa one-hot. Ví dụ: job (12 giá trị) tạo ra 12 cột, month (12 giá trị) tạo ra 12 cột, poutcome (4 giá trị) tạo ra 4 cột... Tổng cộng 9 thuộc tính phân loại sinh ra 44 cột nhị phân (0/1). Cộng với 6 thuộc tính số ban đầu, ta có tổng cộng 50 đặc trưng.

`Câu 2`: Đó là lỗi rò rỉ dữ liệu (data leakage). Hàm fit sẽ tính toán trung bình, độ lệch chuẩn (của StandardScaler) và danh sách giá trị (của OneHotEncoder) dựa trên tập validation. Điều này khiến validation không còn là dữ liệu "chưa từng thấy", làm kết quả đánh giá bị lạc quan giả tạo. Nguyên tắc là chỉ fit trên train và transform trên val.


## 2.3 `[KIẾN THỨC]` Pipeline và trọng số lớp
**Pipeline** ghép tiền xử lý và mô hình thành một khối: `fit(train, ...)` học cả hai trên train; `predict(val)` chỉ biến đổi rồi dự đoán. Hàm `tao_tien_xu_ly()` dưới đây đóng gói bộ tiền xử lý của mục 2.2 để dùng lại.

**Trọng số lớp.** Train có khoảng 12% `yes`. Nếu mỗi lỗi có cùng mức phạt, mô hình "có lợi" khi gần như luôn dự đoán `no`. Tham số `class_weight="balanced"` phạt lỗi trên lớp $c$ với trọng số $w_c = n / (2\,n_c)$: với $n = 27126$, $n_{yes} = 3173$, $n_{no} = 23953$, ta có $w_{yes} \approx 4{,}27$ và $w_{no} \approx 0{,}57$, tức bỏ lỡ một khách `yes` bị phạt nặng gấp khoảng 7,5 lần gọi nhầm một khách `no`.

`[MINH HỌA]`

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

def tao_tien_xu_ly(them_pl=()):
    """Tiền xử lý của mục 2.2: one-hot cho thuộc tính phân loại, chuẩn hóa cho thuộc tính số."""
    return make_column_transformer((OneHotEncoder(handle_unknown="ignore"), THUOC_TINH_PL + list(them_pl)),
                                   (StandardScaler(), THUOC_TINH_SO), verbose_feature_names_out=False)

## 2.4 `[YÊU CẦU]` E1: hồi quy logistic
Yêu cầu sinh viên thực hiện: dựng pipeline `e1` gồm `tao_tien_xu_ly()` và `LogisticRegression(max_iter=1000, class_weight="balanced")`, huấn luyện trên `train` (nhãn `train["y"]`), dự đoán trên `val`, lưu vào `y_val_e1`.

In [7]:
e1 = make_pipeline(tao_tien_xu_ly(), LogisticRegression(max_iter=1000, class_weight="balanced")).fit(train, train["y"])          # YÊU CẦU: make_pipeline(...).fit(train, train["y"])
y_val_e1 = e1.predict(val)    # YÊU CẦU: dự đoán trên val

`[MINH HỌA]` Cùng mô hình nhưng **không** dùng trọng số lớp:

In [8]:
e1_khong_trong_so = make_pipeline(tao_tien_xu_ly(), LogisticRegression(max_iter=1000)).fit(train, train["y"])
pd.DataFrame({"E1 có class_weight": chi_so(val["y"], y_val_e1),
              "E1 không có class_weight": chi_so(val["y"], e1_khong_trong_so.predict(val))}).T.round(3)

,accuracy,precision_yes,recall_yes,f1_yes
E1 có class_weight,0.755,0.267,0.629,0.375
E1 không có class_weight,0.894,0.675,0.188,0.294


## 2.5 `[YÊU CẦU]` E2: k láng giềng gần nhất (kNN)
kNN dự đoán theo đa số nhãn của $k$ khách **gần nhất** trong train. kNN không có tham số `class_weight`.
Yêu cầu sinh viên thực hiện: dựng pipeline `e2` gồm `tao_tien_xu_ly()` và `KNeighborsClassifier(n_neighbors=15)`, huấn luyện trên train, dự đoán trên val, lưu vào `y_val_e2` (khoảng 5 giây).

In [10]:
e2 = make_pipeline(tao_tien_xu_ly(), KNeighborsClassifier(n_neighbors=15)).fit(train, train["y"])          # YÊU CẦU
y_val_e2 = e2.predict(val)    # YÊU CẦU

`[MINH HỌA]` kNN khi **không chuẩn hóa** thuộc tính số (giữ nguyên giá trị gốc):

In [11]:
e2_khong_chuan_hoa = make_pipeline(make_column_transformer((OneHotEncoder(handle_unknown="ignore"), THUOC_TINH_PL), ("passthrough", THUOC_TINH_SO)),
                                   KNeighborsClassifier(n_neighbors=15)).fit(train, train["y"])
pd.DataFrame({"E2 có chuẩn hóa": chi_so(val["y"], y_val_e2),
              "E2 không chuẩn hóa": chi_so(val["y"], e2_khong_chuan_hoa.predict(val))}).T.round(3)

,accuracy,precision_yes,recall_yes,f1_yes
E2 có chuẩn hóa,0.891,0.625,0.170,0.267
E2 không chuẩn hóa,0.884,0.539,0.052,0.095


## 2.6 `[MINH HỌA]` Bảng so sánh trên validation (dùng cho **Bảng 9**)

In [12]:
from sklearn.dummy import DummyClassifier
y_val_e0 = DummyClassifier(strategy="most_frequent").fit(train, train["y"]).predict(val)
ket_qua_val = pd.DataFrame({"E0 lớp đa số": chi_so(val["y"], y_val_e0),
                            "E1 hồi quy logistic": chi_so(val["y"], y_val_e1),
                            "E2 kNN (k = 15)": chi_so(val["y"], y_val_e2)}).T.round(3)
ket_qua_val

,accuracy,precision_yes,recall_yes,f1_yes
E0 lớp đa số,0.883,0.000,0.000,0.000
E1 hồi quy logistic,0.755,0.267,0.629,0.375
E2 kNN (k = 15),0.891,0.625,0.170,0.267


In [13]:
# [TỰ KIỂM TRA]
assert (ket_qua_val.loc[["E1 hồi quy logistic", "E2 kNN (k = 15)"], "f1_yes"] > ket_qua_val.loc["E0 lớp đa số", "f1_yes"]).all(), \
    "CHƯA ĐẠT: E1 và E2 phải có F1-yes lớn hơn E0; cần kiểm tra lại dự đoán."
print("ĐẠT: bảng validation đã đủ E0, E1, E2.")

ĐẠT: bảng validation đã đủ E0, E1, E2.


`[KIẾN THỨC]` **Đọc bảng đúng cách.** Validation có 1.058 khách `yes`; tìm thêm **một** khách làm Recall-yes tăng khoảng 0,001. Chênh lệch 0,01 tương ứng khoảng 10 khách. Ngoài ra, kNN không có trọng số lớp nên ở ngưỡng mặc định gần như chỉ dự đoán `yes` khi đa số láng giềng là `yes` — điều này sẽ được bàn lại ở tuần 3.

## 2.7 `[MINH HỌA]` Rò rỉ dữ liệu: nếu giữ `duration`
Ô dưới đây huấn luyện lại E1 với thêm `duration` (chỉ để đo tác động; đồ án **không** dùng mô hình này). Ba bảng so sánh ở mục 2.4, 2.5 và mục này gộp thành **Bảng 10** của báo cáo.

In [14]:
e1_ro_ri = make_pipeline(make_column_transformer((OneHotEncoder(handle_unknown="ignore"), THUOC_TINH_PL),
                                                 (StandardScaler(), THUOC_TINH_SO + ["duration"])),
                         LogisticRegression(max_iter=1000, class_weight="balanced")).fit(train, train["y"])
pd.DataFrame({"E1 (không có duration)": chi_so(val["y"], y_val_e1),
              "E1 + duration (rò rỉ)": chi_so(val["y"], e1_ro_ri.predict(val))}).T.round(3)

,accuracy,precision_yes,recall_yes,f1_yes
E1 (không có duration),0.755,0.267,0.629,0.375
E1 + duration (rò rỉ),0.845,0.416,0.814,0.551


## 2.8 `[MINH HỌA]` Hồi quy logistic dựa vào những đặc trưng nào?
Mỗi đặc trưng có một hệ số. Trong scikit-learn, `coef_` hướng về lớp `classes_[1]` = `yes`, nên **hệ số dương đẩy về yes**, hệ số âm đẩy về no. Vì các thuộc tính số đã được chuẩn hóa, độ lớn hệ số của chúng so sánh được với nhau (dùng cho **Bảng 11**).

In [15]:
he_so = pd.Series(e1[-1].coef_[0], index=e1[0].get_feature_names_out()).sort_values()
pd.DataFrame({"đẩy về yes": he_so.index[::-1][:10], "hệ số (dương)": he_so.values[::-1][:10].round(2),
              "đẩy về no": he_so.index[:10], "hệ số (âm)": he_so.values[:10].round(2)})

,đẩy về yes,hệ số (dương),đẩy về no,hệ số (âm)
0,poutcome_success,1.65,month_jan,-1.13
1,month_mar,1.23,month_nov,-0.81
2,month_oct,0.90,month_aug,-0.70
3,month_dec,0.88,contact_unknown,-0.67
4,month_sep,0.78,poutcome_failure,-0.64
5,contact_cellular,0.48,month_jul,-0.61
6,job_retired,0.48,poutcome_unknown,-0.54
7,job_student,0.38,job_housemaid,-0.50
8,housing_no,0.28,month_may,-0.41
9,contact_telephone,0.27,poutcome_other,-0.38


`[BÁO CÁO]`
1. Chép bảng validation của E0, E1, E2.
2. Một nhận xét so sánh E1 và E2, có xét Precision và Recall, và có xét tác động của trọng số lớp, của chuẩn hóa (mục 2.4, 2.5).
3. Một đoạn về rò rỉ dữ liệu: F1-yes tăng bao nhiêu khi giữ `duration`; vì sao con số đó không đáng tin.
4. Một nhận xét về các đặc trưng có hệ số mạnh; có phù hợp với các nhận xét EDA ở tuần 1 không.

*Trả lời của sinh viên:*

1. Bảng validation của E0, E1, E2:

- E0 (Lớp đa số): Accuracy 0,883; Precision 0,000; Recall 0,000; F1-yes 0,000.

- E1 (Hồi quy logistic): Accuracy 0,755; Precision 0,267; Recall 0,629; F1-yes 0,375.

- E2 (kNN, k=15): Accuracy 0,891; Precision 0,625; Recall 0,170; F1-yes 0,267.

2. Nhận xét so sánh E1 và E2 (xét Precision, Recall, trọng số lớp, chuẩn hóa):
- Mặc dù E2 có Accuracy (0,891) và Precision (0,625) cao hơn E1, nhưng E1 lại là mô hình tốt hơn vì có F1-yes cao hơn hẳn (0,375 so với 0,267). Nguyên nhân là do E1 được sử dụng class_weight="balanced", giúp mô hình phạt nặng lỗi bỏ lỡ khách hàng tiềm năng, từ đó kéo Recall-yes lên rất cao (0,629 so với 0,170 của E2). Nếu E1 không có trọng số lớp, F1-yes của nó sẽ giảm xuống chỉ còn 0,294 (Recall giảm mạnh xuống 0,188). Đối với E2 (kNN), việc áp dụng chuẩn hóa dữ liệu là cực kỳ quan trọng: nếu không chuẩn hóa, F1-yes của E2 sẽ sụt giảm thảm hại từ 0,267 xuống chỉ còn 0,095 (do khoảng cách bị chi phối bởi biến balance).

3. Đoạn về rò rỉ dữ liệu:
- Khi đưa thuộc tính duration vào mô hình E1, F1-yes trên validation đã tăng vọt từ 0,375 lên 0,551 (Recall tăng từ 0,629 lên 0,814). Tuy nhiên, con số này hoàn toàn không đáng tin cậy.
duration là thời lượng cuộc gọi, chỉ được biết sau khi cuộc gọi kết thúc. Tại thời điểm quyết định có nên gọi hay không, ngân hàng không thể biết thông tin này. Đây là một dạng rò rỉ dữ liệu (data leakage) điển hình, tạo ra điểm số ảo trên giấy tờ nhưng vô dụng trong thực tế.

4. Nhận xét về các đặc trưng có hệ số mạnh:
- Các đặc trưng đẩy mạnh nhất về yes là poutcome_success (hệ số 1,65), month_mar (1,23), month_oct (0,90). Các đặc trưng đẩy mạnh nhất về no là month_jan (-1,13), month_nov (-0,81), contact_unknown (-0,67). Những hệ số này hoàn toàn phù hợp với nhận xét EDA ở Tuần 1: nhóm khách có kết quả chiến dịch trước là success có tỉ lệ đồng ý cao nhất (63,8%); khách liên lạc qua kênh unknown thường ít đồng ý. Mô hình đã học được các tương quan này từ dữ liệu.

`[CÂU HỎI SUY NGẪM]` (Câu 3) Hệ số của `month_mar` dương và lớn. Điều đó có nghĩa "gọi vào tháng 3 làm khách đồng ý" không?

`Câu 3`: Không. Hệ số dương (1,23) cho biết trong dữ liệu quá khứ, các khách được gọi vào tháng 3 có xác suất đồng ý cao hơn khi các yếu tố khác giữ nguyên. Mô hình học được mối tương quan, không phải quan hệ nhân quả. Tháng 3 có thể là thời điểm ngân hàng thực hiện một chiến dịch gọi chọn lọc (nhắm vào khách hàng mục tiêu), nên tỉ lệ thành công cao, chứ không phải bản thân tháng 3 mang lại may mắn.

`[KIẾN THỨC]` **Nhìn trước tuần 3: hồi quy logistic là một nơ-ron.** E1 tính $p_{yes} = \sigma(\mathbf{w}^\top\mathbf{x} + b)$: lấy tổng có trọng số của 50 đặc trưng (các hệ số ở mục 2.8) rồi đưa qua hàm sigmoid $\sigma(a) = 1/(1 + e^{-a})$. Đây đúng là **một nơ-ron** với hàm kích hoạt sigmoid. Vì $\sigma$ đồng biến, khi các đặc trưng khác giữ nguyên, xác suất `yes` chỉ có thể tăng đều hoặc giảm đều theo `age`, nên E1 không biểu diễn được dạng chữ U của Hình 2. Tuần 3 thêm một **lớp nơ-ron ẩn** (mạng MLP) để vượt qua giới hạn này.

## Nộp cuối tuần 2
- [ ] Mọi ô `[TỰ KIỂM TRA]` in **ĐẠT**.
- [ ] Đã có bảng validation E0–E2 và bốn ý của mục `[BÁO CÁO]`.
- [ ] **Viết nháp báo cáo:** Mục 3.1–3.5 (Bảng 7–11); dòng tuần 2 của Phụ lục A; Hình B.2.
- [ ] Tải notebook đã chạy (*File → Download → Download .ipynb*), đặt tên `MSSV_tuan2.ipynb` và **lưu lại**: cả bốn notebook được nộp một lần cùng báo cáo vào cuối tuần 4 (không có đợt nộp giữa kỳ).

---
### Gợi ý trả lời các câu hỏi suy ngẫm
<details><summary>Mở phần này sau khi đã tự trả lời</summary>

**Câu 1.** Mỗi thuộc tính phân loại thành nhiều cột one-hot: 9 thuộc tính phân loại cho 44 cột (job 12, marital 3, education 4, default 2, housing 2, loan 2, contact 3, month 12, poutcome 4), cộng 6 thuộc tính số, tổng 50 đặc trưng.

**Câu 2.** Khi đó trung bình, độ lệch chuẩn (và danh sách giá trị của one-hot) được học từ validation: validation không còn là dữ liệu "chưa thấy", và cùng một giá trị tuổi bị biến đổi khác nhau ở train và validation. Đây là rò rỉ dữ liệu qua tiền xử lý.

**Câu 3.** Không. Hệ số chỉ cho biết trong dữ liệu này, khi các đặc trưng khác giữ nguyên, khách được gọi vào tháng 3 có xác suất đồng ý cao hơn. Mô hình học tương quan, không học quan hệ nhân quả; tháng 3 có thể ứng với một đợt gọi chọn lọc (xem tuần 1, Câu 1).
</details>